In [1]:
import sys, os, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch

REPO_ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))
import data_utils as du
import models as mdl
import train_utils as tu

os.chdir(REPO_ROOT)  # trabajar con rutas relativas a la raiz del repo (idempotente: seguro correr esta celda mas de una vez)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

# Proyecto 3: este notebook es la linea base SECUENCIAL y se ejecuta unicamente en CPU.
# (Se elimino todo uso de CUDA; la version acelerada esta en notebooks/asl_cuda.ipynb.)
DEVICE = torch.device("cpu")
SEQ_THREADS = int(os.getenv("P3_SEQ_THREADS", "0"))  # el benchmark (bash) fija 1 hilo; 0 = valor por defecto de PyTorch
if SEQ_THREADS > 0:
    torch.set_num_threads(SEQ_THREADS)
print(f"Device: {DEVICE} | hilos de PyTorch en CPU: {torch.get_num_threads()}")
print(f"Clases ({len(du.CLASSES)}): {du.CLASSES}")

Device: cpu | hilos de PyTorch en CPU: 1
Clases (29): ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']


## 13. Proyecto 3 — Benchmark secuencial (solo CPU)

Pipeline medido, **todo en CPU y en un solo proceso** (esta es la referencia del speedup):

| Fase | Qué hace | Dónde corre |
|---|---|---|
| 1. Preparación de datos | listar, decodificar JPG, redimensionar (`du.build_subsample`), partir, guardar y leer `.npz` | CPU, **1 proceso** |
| 2. Entrenamiento | `SmallCNN`, Adam, validación en cada época (`tu.train_model`) | CPU, PyTorch con `P3_SEQ_THREADS` hilos (el benchmark usa 1) |
| 3. Evaluación | accuracy y pérdida en el conjunto de test | CPU |

Los parámetros (épocas, batch, lr, imágenes por clase, semilla…) llegan por variables `P3_*` que fijan los scripts bash,
de modo que **la versión CUDA corre con exactamente los mismos datos y condiciones**.

In [2]:
# ---- Proyecto 3 · configuración del benchmark secuencial ----
from bench_utils import cargar_config, RegistroFases

cfg = cargar_config("secuencial")
torch.set_num_threads(cfg.seq_threads)      # 1 = estrictamente secuencial
RAW_DIR = Path(cfg.raw_dir)
reg = RegistroFases(cfg, meta={"torch": torch.__version__, "hilos_torch": torch.get_num_threads(), "dispositivo": "cpu"})
print(cfg)
print(f"Hilos de PyTorch: {torch.get_num_threads()} | dispositivo: {DEVICE}")

ConfigBench(etiqueta='secuencial', arquitectura='SmallCNN', n_per_class=600, img_size=64, epochs=15, batch_size=128, lr=0.001, weight_decay=0.0001, seed=42, workers=1, seq_threads=1, rep=3, raw_dir='C:\\Users\\cr57b\\Documentos\\Proyecto_3_deep_learning\\data\\raw\\asl_alphabet_train\\asl_alphabet_train', processed_dir='C:\\Users\\cr57b\\Documentos\\Proyecto_3_deep_learning\\data\\processed\\bench_secuencial', salida_dir='C:\\Users\\cr57b\\Documentos\\Proyecto_3_deep_learning\\pruebas\\resultados')
Hilos de PyTorch: 1 | dispositivo: cpu


In [3]:
# ---- FASE 1: preparación de datos (CPU, un solo proceso) ----
carpeta = Path(cfg.processed_dir)

with reg.fase("carga_resize", grupo="prep_datos"):
    X_all, y_all, names_all = du.build_subsample(n_per_class=cfg.n_per_class, img_size=cfg.img_size,
                                                 seed=cfg.seed, raw_dir=RAW_DIR)
with reg.fase("split_estratificado", grupo="prep_datos"):
    tr_idx, va_idx, te_idx = du.stratified_split(y_all, seed=cfg.seed)
with reg.fase("guardar_npz", grupo="prep_datos"):
    du.save_processed(X_all[tr_idx], y_all[tr_idx], [names_all[i] for i in tr_idx], "train", carpeta)
    du.save_processed(X_all[va_idx], y_all[va_idx], [names_all[i] for i in va_idx], "val", carpeta)
    du.save_processed(X_all[te_idx], y_all[te_idx], [names_all[i] for i in te_idx], "test", carpeta)
with reg.fase("leer_npz", grupo="prep_datos"):
    X_train, y_train, _ = du.load_processed("train", carpeta)
    X_val, y_val, _ = du.load_processed("val", carpeta)
    X_test, y_test, _ = du.load_processed("test", carpeta)

print(f"train: {X_train.shape}, val: {X_val.shape}, test: {X_test.shape}, dtype={X_train.dtype}")

[FASE] >>> carga_resize (prep_datos)


[FASE] <<< carga_resize: 8.794 s


[FASE] >>> split_estratificado (prep_datos)


[FASE] <<< split_estratificado: 0.001 s


[FASE] >>> guardar_npz (prep_datos)


[FASE] <<< guardar_npz: 5.928 s


[FASE] >>> leer_npz (prep_datos)


[FASE] <<< leer_npz: 0.781 s


train: (12180, 64, 64, 3), val: (2610, 64, 64, 3), test: (2610, 64, 64, 3), dtype=uint8


In [4]:
# ---- FASE 2: entrenamiento (CPU / PyTorch) ----
N_CLASSES = len(du.CLASSES)
torch.manual_seed(cfg.seed)
model_seq = mdl.SmallCNN(N_CLASSES, cfg.img_size)
print(f"Parámetros entrenables: {tu.count_params(model_seq):,}")
train_loader_b, val_loader_b = tu.make_loaders(X_train, y_train, X_val, y_val, batch_size=cfg.batch_size)

with reg.fase("entrenamiento", grupo="entrenamiento"):
    hist_seq = tu.train_model(model_seq, train_loader_b, val_loader_b, DEVICE,
                              epochs=cfg.epochs, lr=cfg.lr, weight_decay=cfg.weight_decay)

Parámetros entrenables: 1,057,533
[FASE] >>> entrenamiento (entrenamiento)


epoch 1/15 train_loss=3.1745 train_acc=0.0984 val_loss=2.6962 val_acc=0.2337 (12.5s)


epoch 2/15 train_loss=2.4479 train_acc=0.2588 val_loss=2.0331 val_acc=0.4050 (12.9s)


epoch 3/15 train_loss=1.9886 train_acc=0.3637 val_loss=1.6617 val_acc=0.4950 (13.0s)


epoch 4/15 train_loss=1.6895 train_acc=0.4424 val_loss=1.4149 val_acc=0.5487 (13.1s)


epoch 5/15 train_loss=1.4878 train_acc=0.5061 val_loss=1.2392 val_acc=0.5962 (13.2s)


... (6 outputs left out)

epoch 12/15 train_loss=0.8487 train_acc=0.7053 val_loss=0.6809 val_acc=0.7812 (14.0s)


epoch 13/15 train_loss=0.7875 train_acc=0.7252 val_loss=0.5751 val_acc=0.8310 (13.4s)


epoch 14/15 train_loss=0.7494 train_acc=0.7398 val_loss=0.6389 val_acc=0.7939 (13.9s)


epoch 15/15 train_loss=0.7079 train_acc=0.7572 val_loss=0.5161 val_acc=0.8425 (13.9s)
[FASE] <<< entrenamiento: 204.236 s


In [5]:
# ---- FASE 3: evaluación en test (CPU) ----
test_loader_b = torch.utils.data.DataLoader(tu.ASLTensorDataset(X_test, y_test), batch_size=cfg.batch_size, shuffle=False)
with reg.fase("evaluacion_test", grupo="evaluacion"):
    test_loss_seq, test_acc_seq = tu.evaluate(model_seq, test_loader_b, DEVICE)
print(f"Test: loss={test_loss_seq:.4f}  accuracy={test_acc_seq:.4f}")

[FASE] >>> evaluacion_test (evaluacion)


[FASE] <<< evaluacion_test: 1.580 s


Test: loss=0.5313  accuracy=0.8483


In [6]:
# ---- Resumen y guardado (lo consume pruebas/generar_reporte.py) ----
reg.guardar(extra={"historial": hist_seq, "test_loss": test_loss_seq, "test_acc": test_acc_seq,
                   "n_parametros": tu.count_params(model_seq)})
resumen = pd.DataFrame(reg.fases)[["fase", "grupo", "duracion_s"]]
display(resumen)
print(f"TOTAL del pipeline secuencial: {reg.total():.2f} s")

[BENCH] resultados guardados en C:\Users\cr57b\Documentos\Proyecto_3_deep_learning\pruebas\resultados\crudo\secuencial_rep3.json


,fase,grupo,duracion_s
0,carga_resize,prep_datos,8.794316
1,split_estratificado,prep_datos,0.001297
2,guardar_npz,prep_datos,5.928059
3,leer_npz,prep_datos,0.781024
4,entrenamiento,entrenamiento,204.236255
5,evaluacion_test,evaluacion,1.580303


TOTAL del pipeline secuencial: 221.32 s
